In [1]:
import os
import sys
import torch
from torch.utils.data import DataLoader
from torchvision.transforms.functional import normalize, to_pil_image
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

parent_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
sys.path.insert(0, parent_dir)

import datasets
from models import get_model
from utils import resize_density_map

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Update path to your best multiclass checkpoint
model_info_path = os.path.join(parent_dir, "checkpoints", "demo_data", "best_mae_0.pth")

model = get_model(model_info_path)
model = model.to(device)
model.eval()
print(f"Model loaded with {model.num_classes} classes.")

c:\Users\QUAN\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model loaded with 2 classes.


In [2]:
dataset_name = "demo_data"
split = "val"
num_classes = model.num_classes

dataset = datasets.Crowd(dataset=dataset_name, split=split, sigma=8, return_filename=True, num_classes=num_classes)
dataloader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0, collate_fn=datasets.collate_fn)
data_iter = iter(dataloader)

# Normalize parameters (standard CLIP/ImageNet)
mean = (0.48145466, 0.4578275, 0.40821073)
std = (0.26862954, 0.26130258, 0.27577711)
alpha = 0.6

Using demo_data
Using custom dataset 'demo_data'. Skipping sanity checks.
Found 68 images and 68 labels for the 'val' split.


In [4]:
from skimage.feature import peak_local_max

try:
    image, points, gt_density, image_name = next(data_iter)
except StopIteration:
    data_iter = iter(dataloader)
    image, points, gt_density, image_name = next(data_iter)

image_height, image_width = image.shape[-2:]
image = image.to(device)

with torch.no_grad():
    # 1. Inference - Get model block-level output
    # shape: (1, num_classes, H_block, W_block)
    pred_den_map_block = model(image)
    
    # 2. Calculate counts per class DIRECTLY from block-level output to be precise
    pred_counts = pred_den_map_block.sum(dim=(2, 3)).squeeze().cpu().numpy()
    if num_classes == 1: pred_counts = np.array([pred_counts])
    
    # 3. Upsample for visualization using the integral-preserving helper
    pred_den_map = resize_density_map(pred_den_map_block, (image_height, image_width))
    
    # 4. GT Counts
    gt_counts = gt_density.sum(dim=(2, 3)).squeeze().cpu().numpy()
    if num_classes == 1: gt_counts = np.array([gt_counts])

# Unnormalize image for visualization
vis_image = normalize(image.clone().cpu(), mean=(0., 0., 0.), std=(1. / std[0], 1. / std[1], 1. / std[2]))
vis_image = normalize(vis_image, mean=(-mean[0], -mean[1], -mean[2]), std=(1., 1., 1.))
vis_image = to_pil_image(vis_image.squeeze(0))

# Visualization
fig, axes = plt.subplots(num_classes, 2, figsize=(12, 5 * num_classes), dpi=150)
if num_classes == 1:
    axes = np.expand_dims(axes, axis=0)

for c in range(num_classes):
    # GT Column
    axes[c, 0].imshow(vis_image)
    axes[c, 0].imshow(gt_density[0, c].cpu().numpy(), cmap="jet", alpha=alpha)
    axes[c, 0].set_title(f"Class {c} GT: {gt_counts[c]:.2f}")
    axes[c, 0].axis("off")
    
    # Prediction Column
    axes[c, 1].imshow(vis_image)
    axes[c, 1].imshow(pred_den_map[0, c].cpu().numpy(), cmap="jet", alpha=alpha)
    axes[c, 1].set_title(f"Class {c} Pred: {pred_counts[c]:.2f}")
    axes[c, 1].axis("off")

plt.tight_layout()
plt.show()

print(f"Image: {image_name[0]}")
for c in range(num_classes):
    print(f"Class {c} -> GT: {gt_counts[c]:.2f}, Pred: {pred_counts[c]:.2f}, Error: {abs(gt_counts[c] - pred_counts[c]):.2f}")
print(f"Total -> GT: {sum(gt_counts):.2f}, Pred: {sum(pred_counts):.2f}, Error: {abs(sum(gt_counts) - sum(pred_counts)):.2f}")

RuntimeError: The size of tensor a (512) must match the size of tensor b (2) at non-singleton dimension 3